# Tellura | Reproducible Compound Aridification Index Trend Analysis

**NASA Space Apps Challenge 2026** — reproducible analysis notebook.

Variable: `T2M` | Dataset: nasa_power

All data fetched live from NASA POWER API — zero synthetic values.


In [ ]:
# Install dependencies
!pip install -q requests numpy scipy pymannkendall matplotlib pandas

In [ ]:
import requests, numpy as np, pandas as pd

POWER_URL = 'https://power.larc.nasa.gov/api/temporal/monthly/regional'

def fetch_nasa_power(param, bbox, start_year, end_year):
    lon_min, lat_min, lon_max, lat_max = bbox
    r = requests.get(POWER_URL, params={
        'parameters': param, 'community': 'RE',
        'longitude-min': lon_min, 'longitude-max': lon_max,
        'latitude-min': lat_min,  'latitude-max': lat_max,
        'start': f'{start_year}01', 'end': f'{end_year}12',
        'format': 'JSON',
    }, timeout=120)
    r.raise_for_status()
    data = r.json()['properties']['parameter'][param]
    years = sorted({int(k[:4]) for k in data})
    annual = {}
    for y in years:
        vals = [v for k, v in data.items() if k.startswith(str(y)) and v != -999]
        if vals:
            annual[y] = np.mean(vals)
    return annual

BBOX   = [-106.6, 25.8, -93.5, 36.5]  # Texas Basin
PARAM  = 'T2M'
annual = fetch_nasa_power(PARAM, BBOX, 2000, 2023)
years  = np.array(sorted(annual))
vals   = np.array([annual[y] for y in years])
print(f'Fetched {len(years)} annual observations from NASA POWER')

In [ ]:
import pymannkendall as mk
from scipy import stats

result = mk.original_test(vals)
slope, intercept, *_ = stats.linregress(years - years[0], vals)

print(f'Mann-Kendall p-value : {result.p:.4f}')
print(f'Sen slope            : {result.slope:+.4f} units/year')
sig = 'YES' if result.p < 0.05 else 'NO'
print(f'Significant (p<0.05) : {sig}')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(years, vals, marker='o', lw=1.5, color='steelblue', label='Annual mean')
trend = slope * (years - years[0]) + intercept
ax.plot(years, trend, '--', color='tomato', label=f'Theil-Sen {slope:+.4f}')
ax.set_xlabel('Year'); ax.set_ylabel(PARAM)
ax.set_title(f'{PARAM} Trend — Texas Basin (NASA POWER, 2000-2023)')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
print('''BibTeX:

@dataset{nasa_power_2024,
  author    = {NASA Langley Research Center},
  title     = {POWER - Prediction of Worldwide Energy Resource, Version 10.0},
  year      = {2024},
  doi       = {10.5067/POWER/10.0.0},
  publisher = {NASA},
  url       = {https://power.larc.nasa.gov}
}

@software{tellura_2026,
  author = {Shivam Gawade},
  title  = {Tellura: Earth System Trend Detective},
  year   = {2026},
  url    = {https://github.com/ShivamGawade-XS/NSC}
}
''')